In [3]:
# Đồ án AI Giao thông – Baseline YOLOv8n trên Vehicles-COCO
#**Điều chỉnh theo yêu cầu thầy Đình Thắng:**
#Chia lại dữ liệu 70:15:15 (thay vì 70/20/10)
#Train 100 epochs + Early Stopping để xem mô hình hội tụ ở epoch nào

In [4]:
# ============================================================
# [CHECK-GPU] Kiểm tra GPU (card, VRAM) được Colab cấp
# ============================================================
!nvidia-smi


Sat Aug  8 02:34:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# ============================================================
# [SETUP] Cài đặt thư viện Ultralytics (YOLOv8)
# ============================================================
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.5 MB/s eta 0:00:00


In [6]:
# ============================================================
# [CHECK] Kiểm tra môi trường Ultralytics (phiên bản, GPU, RAM)
# ============================================================
import ultralytics
ultralytics.checks()

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 46.7/112.6 GB disk)


In [7]:
# ============================================================
# [DRIVE] Kết nối (mount) Google Drive vào Colab
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# ============================================================
# [CHECK-DATA] Kiểm tra & đếm số ảnh mỗi split (train/valid/test) trên Drive
# ============================================================
import os

base = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8"

for split in ["train", "valid", "test"]:
    path = os.path.join(base, split, "images")
    if os.path.exists(path):
        n = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"{split:6s}: {n} ảnh")
    else:
        print(f"{split:6s}: (không tìm thấy thư mục images)")

train : 13300 ảnh
valid : 3798 ảnh
test  : 1900 ảnh


In [9]:
# ============================================================
# [COPY] Sao chép dataset từ Drive về ổ local Colab (train nhanh hơn)
# ============================================================
import shutil, os

# Nguồn: dataset trên Google Drive của bạn
src = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO/Vehicles-coco.v2i.yolov8"
# Đích: ổ local tạm của máy chủ Colab (nhanh)
dst = "/content/vehicles_local"

print("⏳ Đang copy dataset vào ổ local của Colab... (chờ vài phút)")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("✅ Copy xong! Dataset giờ ở:", dst)

for split in ["train", "valid", "test"]:
    p = os.path.join(dst, split, "images")
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f"{split:6s}: {n} ảnh")

⏳ Đang copy dataset vào ổ local của Colab... (chờ vài phút)
✅ Copy xong! Dataset giờ ở: /content/vehicles_local
train : 13300 ảnh
valid : 3798 ảnh
test  : 1900 ảnh


In [10]:
# ============================================================
# [SPLIT] Gom 3 split cũ & chia lại dữ liệu theo tỷ lệ 70:15:15
# ============================================================
import os, shutil, random
from pathlib import Path

SRC   = "/content/vehicles_local"     # bộ gốc (đã copy về local ở cell [COPY])
DST   = "/content/vehicles_7015"      # bộ mới 70:15:15
RATIO = (0.70, 0.15, 0.15)
random.seed(42)                       # cố định seed -> tái lập được
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

# (1) Gom TẤT CẢ cặp (ảnh, nhãn) từ cả 3 split cũ
pairs = []
for sp in ["train", "valid", "test"]:
    idir = Path(SRC) / sp / "images"
    if not idir.exists():
        continue
    for img in idir.iterdir():
        if img.suffix.lower() not in IMG_EXTS:
            continue
        lbl = Path(SRC) / sp / "labels" / (img.stem + ".txt")
        pairs.append((img, lbl if lbl.exists() else None))

print("Tổng ảnh gom được:", len(pairs))

# (2) Trộn ngẫu nhiên & chia 70:15:15
random.shuffle(pairs)
n = len(pairs)
n_tr = int(n * RATIO[0])
n_va = int(n * RATIO[1])
buckets = {
    "train": pairs[:n_tr],
    "valid": pairs[n_tr:n_tr + n_va],
    "test":  pairs[n_tr + n_va:],
}

# (3) Tạo thư mục & copy
for sp in buckets:
    (Path(DST) / sp / "images").mkdir(parents=True, exist_ok=True)
    (Path(DST) / sp / "labels").mkdir(parents=True, exist_ok=True)
for sp, items in buckets.items():
    for img, lbl in items:
        shutil.copy(img, Path(DST) / sp / "images" / img.name)
        if lbl:
            shutil.copy(lbl, Path(DST) / sp / "labels" / (img.stem + ".txt"))

# (4) Báo cáo
print("\n===== KẾT QUẢ CHIA 70:15:15 =====")
for sp in ["train", "valid", "test"]:
    c = len(list((Path(DST) / sp / "images").iterdir()))
    print(f"  {sp:6s}: {c} ảnh ({c*100/n:.1f}%)")

Tổng ảnh gom được: 18998

===== KẾT QUẢ CHIA 70:15:15 =====
  train : 13298 ảnh (70.0%)
  valid : 2849 ảnh (15.0%)
  test  : 2851 ảnh (15.0%)


In [12]:
# ============================================================
# [SAVE-DATA] Lưu bộ dataset đã chia 70:15:15 về Drive
# ============================================================
import shutil, os

# Nguồn: bộ đã chia trên ổ tạm Colab
src = "/content/vehicles_7015"
# Đích: thư mục Dataset trên Drive, tên mới Vehicles-COCO_701515
dst = "/content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515"

print("⏳ Đang lưu bộ 70:15:15 về Drive... (chờ vài phút)")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("✅ Đã lưu xong về Drive:", dst)

# Kiểm tra lại số ảnh mỗi split sau khi lưu
for split in ["train", "valid", "test"]:
    p = os.path.join(dst, split, "images")
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f"{split:6s}: {n} ảnh")

⏳ Đang lưu bộ 70:15:15 về Drive... (chờ vài phút)
✅ Đã lưu xong về Drive: /content/drive/MyDrive/Do an AI giao thong/Dataset/Vehicles-COCO_701515
train : 13298 ảnh
valid : 2849 ảnh
test  : 2851 ảnh


In [13]:
# ============================================================
# [YAML] Tạo file data_7015.yaml (trỏ vào bộ 70:15:15)
# ============================================================
yaml_content = """train: /content/vehicles_7015/train/images
val: /content/vehicles_7015/valid/images
test: /content/vehicles_7015/test/images

nc: 4
names: ['bus', 'car', 'motorcycle', 'truck']
"""
with open("/content/data_7015.yaml", "w") as f:
    f.write(yaml_content)
print("✅ Đã tạo data_7015.yaml")
!cat /content/data_7015.yaml

✅ Đã tạo data_7015.yaml
train: /content/vehicles_7015/train/images
val: /content/vehicles_7015/valid/images
test: /content/vehicles_7015/test/images

nc: 4
names: ['bus', 'car', 'motorcycle', 'truck']


In [14]:
# ============================================================
# [TRAIN] Huấn luyện YOLOv8n – 100 epochs + Early Stopping
# ============================================================
from ultralytics import YOLO

model = YOLO("yolov8n.pt")          # khởi tạo từ trọng số pretrained COCO

results = model.train(
    data="/content/data_7015.yaml", # bộ 70:15:15
    epochs=100,                     # ← thầy yêu cầu: tối đa 100 epochs
    patience=20,                    # ← Early Stopping: dừng nếu 20 epoch không cải thiện
    imgsz=640,
    batch=16,                       # nếu lỗi hết VRAM -> giảm còn 8
    seed=42,                        # cố định để tái lập kết quả
    name="baseline_100epochs_es",   # kết quả: /content/runs/detect/baseline_100epochs_es
)

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_7015.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_100epochs_es, nbs=64, nms=Fa

In [15]:
# ============================================================
# [ANALYZE] Xác định epoch hội tụ từ results.csv
# ============================================================
import pandas as pd

csv = "/content/runs/detect/baseline_100epochs_es/results.csv"
df = pd.read_csv(csv)
df.columns = [c.strip() for c in df.columns]

best_row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
print(f"🔎 Mô hình hội tụ tốt nhất tại epoch: {int(best_row['epoch'])}")
print(f"   mAP@0.5      : {best_row['metrics/mAP50(B)']:.4f}")
print(f"   mAP@0.5:0.95 : {best_row['metrics/mAP50-95(B)']:.4f}")
print(f"   Precision    : {best_row['metrics/precision(B)']:.4f}")
print(f"   Recall       : {best_row['metrics/recall(B)']:.4f}")
print(f"\nTổng epoch đã chạy thực tế: {int(df['epoch'].max())}  (nếu < 100 => Early Stopping đã kích hoạt)")

🔎 Mô hình hội tụ tốt nhất tại epoch: 95
   mAP@0.5      : 0.6794
   mAP@0.5:0.95 : 0.4731
   Precision    : 0.7258
   Recall       : 0.5975

Tổng epoch đã chạy thực tế: 100  (nếu < 100 => Early Stopping đã kích hoạt)


In [16]:
# ============================================================
# [SAVE] Lưu kết quả train về Drive
# ============================================================
import shutil, os

src_100 = "/content/runs/detect/baseline_100epochs_es"
dst_100 = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/baseline_100epochs_es"

if os.path.exists(src_100):
    shutil.copytree(src_100, dst_100, dirs_exist_ok=True)
    print("✅ Đã lưu kết quả 100 epoch về Drive:", dst_100)
    for f in ["weights/best.pt", "results.png", "results.csv", "confusion_matrix.png"]:
        p = os.path.join(dst_100, f)
        print(f"{'✅' if os.path.exists(p) else '❌'} {f}")
else:
    print("❌ Không tìm thấy nguồn:")
    !ls /content/runs/detect/

✅ Đã lưu kết quả 100 epoch về Drive: /content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/baseline_100epochs_es
✅ weights/best.pt
✅ results.png
✅ results.csv
✅ confusion_matrix.png


In [17]:
# ============================================================
# [STATS] Thống kê số lượng đối tượng mỗi lớp - kiểm tra imbalance
# ============================================================
import os
from pathlib import Path
from collections import Counter
import pandas as pd

# Trỏ vào bộ 70:15:15 (thống kê toàn bộ train + valid + test)
BASE  = "/content/vehicles_7015"
NAMES = ['bus', 'car', 'motorcycle', 'truck']   # id: 0=bus,1=car,2=motorcycle,3=truck

img_count = Counter()   # số ẢNH có chứa lớp
obj_count = Counter()   # số ĐỐI TƯỢNG (box) của lớp

for split in ["train", "valid", "test"]:
    lbl_dir = Path(BASE) / split / "labels"
    if not lbl_dir.exists():
        continue
    for txt in lbl_dir.glob("*.txt"):
        lines = txt.read_text().strip().splitlines()
        classes_in_img = set()
        for ln in lines:
            if not ln.strip():
                continue
            cid = int(float(ln.split()[0]))
            obj_count[cid] += 1          # đếm từng box
            classes_in_img.add(cid)
        for cid in classes_in_img:
            img_count[cid] += 1          # mỗi lớp chỉ +1 / ảnh

# Tạo bảng thống kê
total_obj = sum(obj_count.values())
rows = []
for cid, name in enumerate(NAMES):
    rows.append({
        "Lớp": name,
        "Số ảnh chứa lớp": img_count[cid],
        "Số đối tượng": obj_count[cid],
        "Tỷ lệ (%)": round(obj_count[cid] * 100 / total_obj, 2) if total_obj else 0
    })

df = pd.DataFrame(rows).sort_values("Số đối tượng", ascending=False).reset_index(drop=True)
print("========== THỐNG KÊ DATASET (train + valid + test) ==========")
print(df.to_string(index=False))
print(f"\nTổng số đối tượng: {total_obj:,}")

# Đánh giá mất cân bằng
mx = df["Số đối tượng"].max()
mn = df["Số đối tượng"].min()
ratio = mx / mn if mn > 0 else float("inf")
print(f"\nTỷ lệ lớp nhiều nhất / ít nhất = {ratio:.1f} : 1")
if ratio > 5:
    print("⚠️  KẾT LUẬN: Dataset MẤT CÂN BẰNG (imbalance) đáng kể (>5:1).")
elif ratio > 2:
    print("🟡 KẾT LUẬN: Dataset mất cân bằng nhẹ (2–5:1).")
else:
    print("✅ KẾT LUẬN: Dataset tương đối cân bằng (<2:1).")

# (Tùy chọn) Lưu bảng thống kê ra CSV để đưa vào báo cáo
df.to_csv("/content/thongke_dataset.csv", index=False, encoding="utf-8-sig")
print("\n✅ Đã lưu bảng thống kê: /content/thongke_dataset.csv")

========== THỐNG KÊ DATASET (train + valid + test) ==========
       Lớp  Số ảnh chứa lớp  Số đối tượng  Tỷ lệ (%)
       car            12796         45841      63.85
     truck             6396         10438      14.54
motorcycle             3680          9149      12.74
       bus             4144          6363       8.86

Tổng số đối tượng: 71,791

Tỷ lệ lớp nhiều nhất / ít nhất = 7.2 : 1
⚠️  KẾT LUẬN: Dataset MẤT CÂN BẰNG (imbalance) đáng kể (>5:1).

✅ Đã lưu bảng thống kê: /content/thongke_dataset.csv


In [21]:
# ============================================================
# [EVAL] Danh gia mo hinh tren tap TEST
# ============================================================
from ultralytics import YOLO

best_path = "/content/runs/detect/baseline_100epochs_es/weights/best.pt"
model = YOLO(best_path)

metrics_test = model.val(
    data="/content/data_7015.yaml",
    split="test",
    name="eval_test_baseline",
)

print("\n===== KET QUA TREN TAP TEST =====")
print("mAP@0.5      =", round(metrics_test.box.map50, 4))
print("mAP@0.5:0.95 =", round(metrics_test.box.map, 4))
print("Precision    =", round(metrics_test.box.mp, 4))
print("Recall       =", round(metrics_test.box.mr, 4))

print("\n===== mAP@0.5 THEO TUNG LOP =====")
names = model.names
ap50_per_class = metrics_test.box.ap50
class_indices = metrics_test.box.ap_class_index
for i in range(len(class_indices)):
    c = class_indices[i]
    ten_lop = names[c]
    map_lop = round(ap50_per_class[i], 4)
    print(ten_lop, "-> mAP50 =", map_lop)

# Luu ket qua ve Drive
import shutil, os
src = "/content/runs/detect/eval_test_baseline"
dst = "/content/drive/MyDrive/Do an AI giao thong/Baseline/Ketqua_Baseline/eval_test_baseline"
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print("\nDa luu ket qua eval test ve Drive:", dst)

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 30.4±5.4 MB/s, size: 46.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/vehicles_7015/test/labels... 2851 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2851/2851 725.9it/s 3.9s
val: New cache created: /content/vehicles_7015/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 179/179 6.0it/s 29.8s
                   all       2851      10683      0.742      0.607      0.686       0.48
                   bus        656        973      0.799      0.744      0.816       0.66
                   car       1937       6901      0.742      0.582       0.66      0.424
    